In [2]:
import pandas as pd
import sqlite3

# 1. Load the unzipped Kaggle data file
# (Ensure this matches the exact file name in your workspace)
df_raw = pd.read_csv("ecommerce_customer_behavior_2026.csv")

# Clean column headers by replacing spaces/special characters with underscores
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# 2. Establish connection to local SQLite database file
conn = sqlite3.connect("ecommerce_behavior.db")
cursor = conn.cursor()

# 3. Write data to a relational SQL table
df_raw.to_sql("customer_analytics", conn, if_exists="replace", index=False)
print("📦 Relational database engine successfully created as 'ecommerce_behavior.db'!")
print(f"Loaded {df_raw.shape[0]} customer records into the 'customer_analytics' table.")

📦 Relational database engine successfully created as 'ecommerce_behavior.db'!
Loaded 100000 customer records into the 'customer_analytics' table.


In [4]:
# Updated SQL query using the precise column definitions from your unzipped dataset schema
rfm_query = """
WITH rfm_base AS (
    SELECT 
        customer_id,
        recency_days,
        total_spending_usd,
        total_items_purchased
    FROM customer_analytics
),
rfm_scores AS (
    SELECT *,
        -- NTILE divides data into 5 equal buckets (1-5 rank matrices)
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY total_items_purchased ASC) AS f_score,
        NTILE(5) OVER (ORDER BY total_spending_usd ASC) AS m_score
    FROM rfm_base
),
rfm_concatenated AS (
    SELECT *,
        (r_score || f_score || m_score) AS rfm_cell_id
    FROM rfm_scores
)
SELECT *,
    CASE
        WHEN rfm_cell_id IN ('555', '554', '544', '454', '455', '445') THEN 'Champions'
        WHEN rfm_cell_id IN ('543', '444', '435', '355', '354', '345', '344', '335') THEN 'Loyal Customers'
        WHEN rfm_cell_id IN ('553', '551', '552', '541', '542', '453', '452', '451') THEN 'Recent/New Ingestions'
        WHEN rfm_cell_id IN ('512', '511', '422', '421', '412', '411', '311') THEN 'Promising/New'
        WHEN rfm_cell_id IN ('532', '531', '432', '431', '332', '322', '321') THEN 'Needs Attention'
        WHEN rfm_cell_id IN ('331', '321', '312', '221', '213', '231', '241', '251') THEN 'About to Churn'
        WHEN rfm_cell_id IN ('255', '254', '245', '244', '253', '252', '243', '242', '235', '234') THEN 'High Risk / Cant Lose Them'
        WHEN rfm_cell_id IN ('155', '154', '145', '144', '214', '215', '115', '114') THEN 'At Severe Risk'
        ELSE 'Lost / Hibernating'
    END AS customer_behavior_segment
FROM rfm_concatenated;
"""

# Extract the calculated segments back into a Pandas dataframe seamlessly
df_segmented = pd.read_sql_query(rfm_query, conn)

print("⚡ SQL RFM Segmentation Complete!")
print("\nBehavioral Segment Volume Counts:")
print(df_segmented['customer_behavior_segment'].value_counts())

# Save segmented output to local directory
df_segmented.to_csv("rfm_customer_segments.csv", index=False)


⚡ SQL RFM Segmentation Complete!

Behavioral Segment Volume Counts:
customer_behavior_segment
Lost / Hibernating            41714
Promising/New                 15213
Champions                     12424
Loyal Customers               11307
High Risk / Cant Lose Them     7987
At Severe Risk                 7305
Needs Attention                4050
Name: count, dtype: int64
